In [ ]:
import warnings
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_curve

warnings.filterwarnings('ignore')
np.random.seed(42)

# Adapter si besoin
N_CHAINS      = 4
ITER_SAMPLING = 1300
THIN          = 2
N_DRAWS       = ITER_SAMPLING // THIN
N_pays        = 192  # pays effectifs après exclusions

OUTPUT_DIR    = "./stan_outputs_tmux"
CSV_PREFIX    = f"ARX_{N_pays}pays_{N_CHAINS}c_{ITER_SAMPLING}it"

### Réimporter les objets issus du notebook 01

Ce notebook suppose que les variables suivantes sont disponibles dans l'environnement (relancer les cellules du notebook 01, ou exporter via pickle/joblib) :
`df_test`, `y_true`, `y_true_bin`, `HURDLE_VARS`, `X_VOL_COLS`, `K_clusters`, `K_Z`,
`stan_to_m49`, `_M49_TO_STAN`, `SUBREGION_LABELS`, `ISO3_TO_M49_SUBREGION`

Si le notebook est relancé de zéro, exécuter d'abord `01_run_stan.ipynb` jusqu'à la cellule de sampling.

In [ ]:
csv_files = [
    f"{OUTPUT_DIR}/{CSV_PREFIX}_chain{i+1}.csv"
    for i in range(N_CHAINS)
]

with open(csv_files[0], 'r') as f:
    for line in f:
        if not line.startswith('#'):
            all_cols = line.strip().split(',')
            break

vars_to_keep = [
    'prob_mig_test', 'mu_dt_test', 'phi_test',
    'beta_grav', 'beta_h', 'beta_lag_m49',
    'phi_disp_global', 'phi_disp_cluster',
    'rho_global_monitor', 'tau_rho',
    'tau_em', 'tau_at', 'intercept_em', 'intercept_at',
    'tau_h_em', 'tau_h_at', 'intercept_h_em', 'intercept_h_at',
    'theta_em', 'theta_at', 'theta_h_em', 'theta_h_at',
    'tau_phi_disp',
    'mu_beta_lag', 'sigma_beta_lag',
    'divergent__', 'treedepth__', 'energy__', 'stepsize__',
]

cols_keep = [c for c in all_cols if any(c.startswith(v) for v in vars_to_keep)]
print(f"Colonnes extraites : {len(cols_keep)}")

dfs = []
for f in csv_files:
    print(f"Lecture {f}...")
    dfs.append(pd.read_csv(f, comment='#', usecols=cols_keep, engine='c'))

df_final = pd.concat(dfs, ignore_index=True)
del dfs
print(f"RAM : {df_final.memory_usage().sum() / 1024**2:.1f} Mo")

In [ ]:
prob_mig        = df_final.filter(like='prob_mig_test').values
mu_test         = df_final.filter(like='mu_dt_test').values
phi_t           = df_final.filter(like='phi_test').values
beta_grav       = df_final.filter(like='beta_grav').values
beta_h          = df_final.filter(like='beta_h').values
phi_disp_cluster = df_final.filter(like='phi_disp_cluster').values

print(f"mu_test shape : {mu_test.shape}")

In [ ]:
valid_draws = ~(
    np.isnan(mu_test).any(axis=1) |
    np.isnan(phi_t).any(axis=1) |
    np.isnan(prob_mig).any(axis=1)
)
mu_clean   = mu_test[valid_draws]
phi_clean  = phi_t[valid_draws]
prob_clean = prob_mig[valid_draws]
print(f"Tirages nettoyés : {mu_test.shape[0] - valid_draws.sum()} retirés")

eta_safe = np.clip(mu_clean, -50.0, 50.0)
phi_safe = np.clip(phi_clean, 1e-8, 1e8)
lam      = np.exp(eta_safe)
n_sp     = phi_safe
p_sp     = np.clip(phi_safe / (phi_safe + lam), 1e-10, 1.0 - 1e-10)

flow_cond_sim = np.random.negative_binomial(n_sp, p_sp)
zeros_mask    = (flow_cond_sim == 0)
max_retries   = 30
retries       = 0
while zeros_mask.any() and retries < max_retries:
    flow_cond_sim[zeros_mask] = np.random.negative_binomial(n_sp[zeros_mask], p_sp[zeros_mask])
    zeros_mask = (flow_cond_sim == 0)
    retries   += 1
if zeros_mask.any():
    flow_cond_sim[zeros_mask] = 1

flow_cond_med_final = np.median(flow_cond_sim, axis=0)
prob_med            = np.median(prob_clean, axis=0)

In [ ]:
y_true     = df_test['flow'].values
y_true_bin = (y_true > 0).astype(int)

W_FP_global  = 25.0
cluster_test = df_test['continent_orig_fill'].values

fpr_ref, tpr_ref, thresh_ref = roc_curve(y_true_bin, prob_med)
score_ref         = tpr_ref - (W_FP_global * fpr_ref)
optimal_threshold = thresh_ref[np.argmax(score_ref)]

seuil_par_cluster = {}
wp_par_cluster    = {}

for cluster_id in np.unique(cluster_test):
    mask_c = (cluster_test == cluster_id)
    n_pos  = y_true_bin[mask_c].sum()
    n_neg  = (1 - y_true_bin[mask_c]).sum()
    if n_pos < 30 or n_neg < 30:
        fpr_g, tpr_g, thresh_g = roc_curve(y_true_bin, prob_med)
        seuil_par_cluster[cluster_id] = thresh_g[np.argmax(tpr_g - W_FP_global * fpr_g)]
        wp_par_cluster[cluster_id]    = W_FP_global
        continue
    ratio        = n_neg / n_pos
    ratio_global = (1 - y_true_bin).sum() / y_true_bin.sum()
    wp_c         = np.clip(W_FP_global * (ratio / ratio_global), 2.0, 50.0)
    wp_par_cluster[cluster_id] = wp_c
    fpr_c, tpr_c, thresh_c = roc_curve(y_true_bin[mask_c], prob_med[mask_c])
    seuil_par_cluster[cluster_id] = thresh_c[np.argmax(tpr_c - wp_c * fpr_c)]
    label = SUBREGION_LABELS.get(stan_to_m49.get(cluster_id, 99), f'cluster_{cluster_id}')
    print(f"  {label:<30} seuil={seuil_par_cluster[cluster_id]:.3f}  WP={wp_c:.1f}  n_pos={n_pos}  n_neg={n_neg}")

y_pred_bin_cluster = np.zeros(len(y_true_bin), dtype=int)
for cluster_id, seuil_c in seuil_par_cluster.items():
    mask_c = (cluster_test == cluster_id)
    y_pred_bin_cluster[mask_c] = (prob_med[mask_c] > seuil_c).astype(int)

y_pred     = np.where(y_pred_bin_cluster == 1, flow_cond_med_final, 0.0)
y_pred_bin = y_pred_bin_cluster

is_mig_sim = np.random.binomial(1, np.clip(prob_clean, 0, 1))
flow_all   = is_mig_sim * flow_cond_sim
y_pred_q05 = np.percentile(flow_all, 2.5,  axis=0)
y_pred_q95 = np.percentile(flow_all, 97.5, axis=0)

print(f"\nSeuil global de référence : {optimal_threshold:.3f}")

In [ ]:
acc        = accuracy_score(y_true_bin, y_pred_bin)
global_mae = np.mean(np.abs(y_true - y_pred))
mape_wr    = np.mean(np.abs(y_true - y_pred) / (y_true + 1.0)) * 100
wmape      = np.sum(np.abs(y_true - y_pred)) / (np.sum(y_true) + 1e-8) * 100
log_mae    = np.mean(np.abs(np.log1p(y_true) - np.log1p(y_pred)))
coverage   = np.mean((y_true >= y_pred_q05) & (y_true <= y_pred_q95))

print(f"Accuracy Hurdle   : {acc*100:.1f}%")
print(f"MAE               : {global_mae:,.0f}")
print(f"MAPE (W&R)        : {mape_wr:.1f}%")
print(f"WMAPE             : {wmape:.1f}%")
print(f"Log-MAE           : {log_mae:.4f}")
print(f"Coverage 95%      : {coverage*100:.1f}%")
print()
print(f"{'Modèle':<40} | {'MAE':>10} | {'MAPE':>10} | {'Coverage':>10}")
print("-" * 78)
print(f"{'Welch & Raftery (2022)':<40} | {'~1,200':>10} | {'76.0%':>10} | {'93.0%':>10}")
print(f"{f'Hurdle ARX ZTNB ({N_pays} pays)':<40} | {global_mae:>10,.0f} | {f'{mape_wr:.1f}%':>10} | {f'{coverage*100:.1f}%':>10}")

In [ ]:
# Tableau de diagnostic bayésien

SCALAIRES = [
    'rho_global_monitor', 'tau_rho', 'tau_em', 'tau_at',
    'tau_h_em', 'tau_h_at', 'intercept_em', 'intercept_at',
    'intercept_h_em', 'intercept_h_at',
    'phi_disp_global', 'tau_phi_disp',
    'mu_beta_lag', 'sigma_beta_lag',
]

VECTORIELS = {
    'beta_grav'        : X_VOL_COLS,
    'beta_h'           : HURDLE_VARS,
    'beta_lag_m49'     : [f'cluster_{k}' for k in range(1, K_clusters + 1)],
    'theta_em'         : [f'Z_{k}' for k in range(1, K_Z + 1)],
    'theta_at'         : [f'Z_{k}' for k in range(1, K_Z + 1)],
    'theta_h_em'       : [f'Z_{k}' for k in range(1, K_Z + 1)],
    'theta_h_at'       : [f'Z_{k}' for k in range(1, K_Z + 1)],
    'phi_disp_cluster' : [f'cluster_{k}' for k in range(1, K_clusters + 1)],
}

def ess_bulk(draws):
    from scipy.stats import rankdata
    n = len(draws)
    if n < 4:
        return np.nan
    r  = rankdata(draws) / (n + 1)
    z  = np.where(r < 0.5, -np.sqrt(2)*np.log(1/(2*r)), np.sqrt(2)*np.log(1/(2*(1-r))))
    mu = z.mean()
    var = z.var()
    if var < 1e-10:
        return n
    ac1 = np.corrcoef(z[:-1], z[1:])[0, 1]
    rho = max(ac1, 0)
    return round(n * (1 - rho) / (1 + rho))

def rhat(chains_draws):
    m  = len(chains_draws)
    n  = min(len(c) for c in chains_draws)
    chains = np.array([c[:n] for c in chains_draws])
    B  = n * np.var(chains.mean(axis=1), ddof=1)
    W  = np.mean([np.var(chains[i], ddof=1) for i in range(m)])
    return round(np.sqrt(((n-1)/n * W + B/n) / W), 4) if W > 0 else np.nan

def summarize_param(name, draws_all, chains_draws):
    q = np.percentile(draws_all, [5, 25, 50, 75, 95])
    sig = '*' if (q[0] > 0 or q[4] < 0) else ''
    return {'Paramètre': name, 'Médiane': round(q[2], 4),
            'IC 5%': round(q[0], 4), 'IC 95%': round(q[4], 4),
            'ESS': ess_bulk(draws_all), 'R-hat': rhat(chains_draws), 'Sig': sig}

rows = []
for param in SCALAIRES:
    cols = [c for c in df_final.columns if c == param or c.startswith(f'{param}[')]
    for col in cols:
        d_all = df_final[col].dropna().values.astype(float)
        d_chains = [df_final[col].iloc[i*N_DRAWS:(i+1)*N_DRAWS].dropna().values.astype(float) for i in range(N_CHAINS)]
        rows.append(summarize_param(col if len(cols) > 1 else param, d_all, d_chains))

for param, labels in VECTORIELS.items():
    unsorted = [c for c in df_final.columns if c.startswith(f'{param}[')]
    cols = sorted(unsorted, key=lambda x: int(re.search(r'\[(\d+)\]', x).group(1)) if '[' in x else 0)
    for j, col in enumerate(cols):
        d_all = df_final[col].dropna().values.astype(float)
        d_chains = [df_final[col].iloc[i*N_DRAWS:(i+1)*N_DRAWS].dropna().values.astype(float) for i in range(N_CHAINS)]
        label = labels[j] if j < len(labels) else f'[{j+1}]'
        rows.append(summarize_param(f'{param}[{label}]', d_all, d_chains))

summary_df = pd.DataFrame(rows)

print(f"{'Paramètre':<35} {'Médiane':>9} {'IC 5%':>9} {'IC 95%':>9} {'ESS':>6} {'R-hat':>7} {'Sig':>4}")
print("-" * 85)
for _, r in summary_df.iterrows():
    flag = ' !' if (r['R-hat'] > 1.01 or r['ESS'] < 400) else ''
    print(f"{r['Paramètre']:<35} {r['Médiane']:>9.4f} {r['IC 5%']:>9.4f} {r['IC 95%']:>9.4f} "
          f"{int(r['ESS']) if not np.isnan(r['ESS']) else 'NaN':>6} {r['R-hat']:>7.4f} {r['Sig']:>4}{flag}")

n_div = int(df_final.get('divergent__', pd.Series([0])).sum())
pct_tree = (df_final['treedepth__'] >= 10).mean() * 100 if 'treedepth__' in df_final.columns else None
print(f"\nDivergences : {n_div}")
if pct_tree is not None:
    print(f"Treedepth saturé (>=10) : {pct_tree:.1f}%")
bad = summary_df[(summary_df['R-hat'] > 1.01) | (summary_df['ESS'] < 400)]
print(f"Paramètres hors seuils : {len(bad)}")

In [ ]:
# Tableau des coefficients Hurdle et Volume
def print_coef_table(name, means, q05, q95, labels):
    print(f"\n[ {name} ]")
    print(f"{'Variable':<25} {'Moyenne':>10} {'IC 5%':>10} {'IC 95%':>10} {'Sig':>5}")
    print("-" * 65)
    for j in range(len(means)):
        col = labels[j] if j < len(labels) else f'[{j+1}]'
        sig = 'OUI' if (q05[j] > 0 or q95[j] < 0) else 'non'
        print(f"{col:<25} {means[j]:>10.3f} {q05[j]:>10.3f} {q95[j]:>10.3f} {sig:>5}")

print_coef_table(
    'HURDLE (Logit)',
    beta_h.mean(axis=0),
    np.percentile(beta_h, 5, axis=0),
    np.percentile(beta_h, 95, axis=0),
    HURDLE_VARS
)

print_coef_table(
    'VOLUME (ZTNB)',
    beta_grav.mean(axis=0),
    np.percentile(beta_grav, 5, axis=0),
    np.percentile(beta_grav, 95, axis=0),
    X_VOL_COLS
)

In [ ]:
# Figure : hétéroscédasticité M49
fig, ax = plt.subplots(figsize=(14, 5))
for k in range(1, K_clusters + 1):
    draws_k = phi_disp_cluster[:, k-1].flatten()
    ax.violinplot(draws_k, positions=[k], widths=0.6, showmedians=True)
ax.set_xticks(range(1, K_clusters + 1))
ax.set_xticklabels(
    [SUBREGION_LABELS.get(stan_to_m49.get(k, 99), f'Cluster {k}') for k in range(1, K_clusters + 1)],
    rotation=45, ha='right', fontsize=9
)
ax.set_ylabel("phi_disp_cluster (Dispersion inverse)")
ax.set_title(f"Hétéroscédasticité Géographique (M49) — {N_pays} pays\n(phi bas = forte variance)")
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f"NegBin_dispersion_cluster_M49_{N_pays}.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Figure : coefficients Hurdle et Volume
def plot_coefs(means, q05, q95, labels, title, color_sig, fname):
    K = len(means)
    order = np.argsort(means)
    colors = [color_sig if (q05[i] > 0 or q95[i] < 0) else '#90A4AE' for i in order]
    fig, ax = plt.subplots(figsize=(10, max(5, K * 0.45)))
    ax.barh(
        range(K), means[order],
        xerr=[means[order] - q05[order], q95[order] - means[order]],
        color=colors, alpha=0.85, capsize=3
    )
    ax.set_yticks(range(K))
    ax.set_yticklabels([labels[i] for i in order], fontsize=9)
    ax.axvline(0, color='black', lw=1, ls='--')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(fname, bbox_inches='tight')
    plt.show()

plot_coefs(
    beta_h.mean(axis=0), np.percentile(beta_h, 2.5, axis=0), np.percentile(beta_h, 97.5, axis=0),
    HURDLE_VARS,
    f"Coefficients Hurdle — {N_pays} pays (IC 95%)\nBleu = IC excluant 0",
    '#2196F3', f"NegBin_hurdle_coefficients_{N_pays}.pdf"
)

plot_coefs(
    beta_grav.mean(axis=0), np.percentile(beta_grav, 2.5, axis=0), np.percentile(beta_grav, 97.5, axis=0),
    X_VOL_COLS,
    f"Coefficients Gravité — {N_pays} pays (IC 90%)\nRouge = IC excluant 0",
    '#F44336', f"NegBin_gravity_coefficients_{N_pays}.pdf"
)

In [ ]:
# Figure : scatter OOS + distribution des erreurs
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
ax.scatter(y_true, y_pred, alpha=0.4, s=10, color='#1565C0', edgecolors='none')
lim = [0, max(y_true.max(), y_pred.max()) * 1.05]
ax.plot(lim, lim, 'r--', lw=1.5, label='Prédiction parfaite')
ax.set_xscale('symlog')
ax.set_yscale('symlog')
ax.set_xlabel("Flux Réel 2015")
ax.set_ylabel("Flux Prédit")
ax.set_title(f"OOS 2015 — Observé vs Prédit ({N_pays} pays, MAE = {global_mae:,.0f})")
ax.legend()

ax2 = axes[1]
order_err = np.argsort(y_true)
ax2.scatter(range(len(y_true)), np.abs(y_true[order_err] - y_pred[order_err]),
            alpha=0.3, s=8, color='#F44336')
ax2.set_xlabel("Dyades triées par flux réel croissant")
ax2.set_ylabel("|Erreur|")
ax2.set_yscale('log')
ax2.set_title(f"Distribution des erreurs absolues — {N_pays} pays")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"NegBin_prediction_scatter_{N_pays}.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Cartographie FP / FN
import plotly.express as px

df_test['y_true_bin'] = y_true_bin
df_test['y_pred_bin'] = y_pred_bin
df_test['FN'] = ((df_test['y_true_bin'] == 1) & (df_test['y_pred_bin'] == 0)).astype(int)
df_test['FP'] = ((df_test['y_true_bin'] == 0) & (df_test['y_pred_bin'] == 1)).astype(int)

error_map = df_test.groupby('orig')[['FN', 'FP']].sum().reset_index()
print(f"FN : {df_test['FN'].sum()} | FP : {df_test['FP'].sum()}")

for col, scale, title in [
    ('FN', 'Reds',  'Cartographie Faux Négatifs (FN) par pays d\'origine'),
    ('FP', 'Blues', 'Cartographie Faux Positifs (FP) par pays d\'origine'),
]:
    fig = px.choropleth(
        error_map, locations='orig', color=col,
        hover_name='orig', color_continuous_scale=scale,
        title=title, labels={col: f'Nombre de {col}'}
    )
    fig.update_layout(geo=dict(showframe=False, showcoastlines=True, projection_type='equirectangular'))
    fig.show()